# Promotion attack: results split by replay mismatch

Read-only analysis of one completed live run from `promotion_attack.ipynb` or `promotion_attack_realtimeqa.ipynb`. It loads no model and needs no GPU or repo code.

- Uses the first candidate per question (`success@1`).
- Splits **uncertified** questions (attack targets) from **certified** questions (controls; RobustRAG guarantees one injected passage cannot flip them).
- Reports each group twice: all questions, and excluding questions whose certification-replay clean answer differed in wording from the logged run (`certification_replay_answer_mismatch`; the 7 RealtimeQA questions, none on NQ).
- Saves the table as a new file `analysis-replay-mismatch-split.json` inside the live run folder; it refuses to overwrite an existing one.


In [ ]:
# Settings: which completed live run to analyse.
# >>> EDIT: replace EDIT-ME with the folders between MyDrive and adversarial_rag-shared in your Drive.
SHARED_PATH = "/content/drive/MyDrive/EDIT-ME/adversarial_rag-shared"
OUTPUT_FOLDER = "manya"
# >>> EDIT: folder name of the live run printed by the attack notebook, under outputs/manya/promotion-attack/.
LIVE_RUN_NAME = "promotion-live-realtimeqa-1+2-gold_alias_substitution-EDIT-ME"


In [ ]:
# Mount Drive and load the live run, its selection run, and per-question flags.
from google.colab import drive
drive.mount("/content/drive")

import json
from pathlib import Path

ATTACK_RUNS_DIR = Path(SHARED_PATH) / "outputs" / OUTPUT_FOLDER / "promotion-attack"
LIVE_RUN = ATTACK_RUNS_DIR / LIVE_RUN_NAME
live_manifest = json.loads((LIVE_RUN / "manifest.json").read_text())
if live_manifest["status"] != "complete":
    raise ValueError(f"Live run is not complete: {LIVE_RUN}")
SELECTION_RUN = ATTACK_RUNS_DIR / Path(live_manifest["environment"]["selection_run"]).name
selection_manifest = json.loads((SELECTION_RUN / "manifest.json").read_text())
selections = json.loads((SELECTION_RUN / "selections.json").read_text())
certification = selection_manifest.get("environment", {}).get("certification") or {}
ALLOWED_FINAL_ANSWER_MISMATCHES = certification.get("allowed_final_answer_mismatches", [])

outcomes_by_index = {outcome["query_index"]: outcome for outcome in selections}
results = []
for path in sorted((LIVE_RUN / "results").glob("query-*.json")):
    first = json.loads(path.read_text())["results"][0]
    outcome = outcomes_by_index[first["query_index"]]
    results.append({**first, "certified": outcome["certified"],
                    "replay_mismatch": bool(outcome.get("certification_replay_answer_mismatch"))})

print("Live run:", LIVE_RUN)
print("Selection run:", SELECTION_RUN)
print("Config:", live_manifest["config"])
print("GPU:", live_manifest["environment"]["gpu"])
print("Allowed replay mismatches:", ALLOWED_FINAL_ANSWER_MISMATCHES)
print("Questions attacked:", len(results))


In [ ]:
# Count each metric for targets and controls, with and without the replay-mismatch questions.
METRICS = ["response_realized", "promotion_realized", "attacked_gold_retained",
           "automatic_target_present", "automatic_gold_present",
           "automatic_strict_targeted_success"]
table = {}
for group, certified in (("uncertified_targets", False), ("certified_controls", True)):
    for subset, keep in (("all", lambda r: True),
                         ("excluding_replay_mismatches", lambda r: not r["replay_mismatch"])):
        rows = [r for r in results if r["certified"] is certified and keep(r)]
        table[f"{group}/{subset}"] = {"n": len(rows), **{m: sum(r[m] for r in rows) for m in METRICS}}

columns = ["n", *METRICS]
print(f"{'group/subset':45}" + "".join(f"{m.removeprefix('automatic_'):>28}" for m in columns))
for name, counts in table.items():
    print(f"{name:45}" + "".join(f"{counts[m]:>28}" for m in columns))
print()
for r in results:
    print(f"q{r['query_index']:>3} certified={r['certified']!s:5} replay_mismatch={r['replay_mismatch']!s:5} "
          f"target={r['target']!r:30} strict={r['automatic_strict_targeted_success']!s:5} "
          f"final={r['attacked_final_answer'][:80]!r}")


In [ ]:
# Save the table next to the live run (never overwrites) and unmount Drive.
with (LIVE_RUN / "analysis-replay-mismatch-split.json").open("x") as destination:
    json.dump({"live_run": str(LIVE_RUN), "selection_run": str(SELECTION_RUN),
               "allowed_final_answer_mismatches": ALLOWED_FINAL_ANSWER_MISMATCHES,
               "candidate_rank_used": 1, "table": table}, destination, indent=2)
drive.flush_and_unmount()
